In [2]:
import os

import cv2
import numpy as np
import pandas as pd
import torch
from natsort import natsorted
from pathlib import Path
from ultralytics import YOLO
import time

from code_programm.path import get_path_weight_model

In [3]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name())
model = YOLO(get_path_weight_model('line_recognition.pt'))

1
NVIDIA GeForce GTX 1080 Ti


In [4]:
wheel_ets_train_x = pd.DataFrame(columns=[i for i in range(160 * 100)])

In [5]:
folder_path = Path('D:\Dataset_for_autopilot')
files_and_folders = os.listdir(folder_path)

# Фильтруем только папки
folders = [f for f in files_and_folders if os.path.isdir(os.path.join(folder_path, f))]

# Сортируем папки по дате изменения
sorted_folders = sorted(folders, key=lambda x: os.path.getmtime(os.path.join(folder_path, x)), reverse=True)

# Выводим список папок
print("Папки в папке {} отсортированы по дате изменения: ".format(folder_path))
for folder in sorted_folders:
    print(folder)

Папки в папке D:\Dataset_for_autopilot отсортированы по дате изменения: 
new
2024-04-01 08-19-11
2024-04-01 08-18-41
2024-04-01 08-18-29
2024-04-01 08-18-18
2024-04-01 08-15-21
2024-04-01 08-13-16
2024-04-01 08-12-10
2024-04-01 08-10-13
2024-04-01 08-08-58
2024-04-01 08-07-13
2024-04-01 08-05-14
2024-04-01 08-03-03
2024-04-01 08-01-05
2024-04-01 07-58-48
2024-04-01 07-55-54
2024-04-01 07-51-15
2024-04-01 07-47-21
2024-04-01 07-42-36
2024-04-01 07-41-03
2024-04-01 07-36-51
2024-04-01 07-34-54
2024-04-01 07-32-35
2024-04-01 07-29-56
2024-04-01 07-26-37
num
2024-03-31 03-29-46
2024-03-31 03-28-05
2024-03-31 03-26-14
2024-03-31 03-24-22
2024-03-31 03-22-07
2024-03-31 03-17-11
2024-03-31 03-11-36
2024-03-31 03-05-57
numbers
Новая папка
2024-03-29 07-04-10
2024-03-29 07-01-32
2024-03-29 06-57-45
2024-03-28 05-07-19
2024-03-27 00-11-26
2024-03-27 00-00-05
2024-02-29 19-40-13
2024-02-29 17-28-08
2024-02-29 15-46-10
2024-02-27 06-09-465
2024-02-27 04-45-43
2024-02-27 03-56-41
2024-02-27 03-51-4

In [6]:
paths = [r'D:\Dataset_for_autopilot\2024-04-01 08-19-11',
         r'D:\Dataset_for_autopilot\2024-04-01 08-18-41',
         r'D:\Dataset_for_autopilot\2024-04-01 08-18-29',
         r'D:\Dataset_for_autopilot\2024-04-01 08-18-18',
         r'D:\Dataset_for_autopilot\2024-04-01 08-15-21',
         r'D:\Dataset_for_autopilot\2024-04-01 08-13-16',
         r'D:\Dataset_for_autopilot\2024-04-01 08-12-10',
         r'D:\Dataset_for_autopilot\2024-04-01 08-10-13',
         r'D:\Dataset_for_autopilot\2024-04-01 08-08-58',
         r'D:\Dataset_for_autopilot\2024-04-01 08-07-13',
         r'D:\Dataset_for_autopilot\2024-04-01 08-05-14',
         r'D:\Dataset_for_autopilot\2024-04-01 08-03-03',
         r'D:\Dataset_for_autopilot\2024-04-01 08-01-05',
         r'D:\Dataset_for_autopilot\2024-04-01 07-58-48',
         r'D:\Dataset_for_autopilot\2024-04-01 07-55-54',
         r'D:\Dataset_for_autopilot\2024-04-01 07-51-15',
         r'D:\Dataset_for_autopilot\2024-04-01 07-47-21',
         r'D:\Dataset_for_autopilot\2024-04-01 07-42-36',
         r'D:\Dataset_for_autopilot\2024-04-01 07-41-03',
         r'D:\Dataset_for_autopilot\2024-04-01 07-36-51',
         r'D:\Dataset_for_autopilot\2024-04-01 07-34-54',
         r'D:\Dataset_for_autopilot\2024-04-01 07-32-35',
         r'D:\Dataset_for_autopilot\2024-04-01 07-29-56',
         r'D:\Dataset_for_autopilot\2024-04-01 07-26-37']

In [7]:
for num_path in paths:
    path_i = os.path.join(num_path, f'road')
    if os.path.exists(f'{path_i}') and os.path.isdir(f'{path_i}'):
        png_files = [os.path.join(path_i, file) for file in os.listdir(path_i) if file.endswith('.png')]
        print("Полные пути к файлам в папке:")
        png_files = natsorted(png_files)
        print(png_files[0], '\n')
    else:
        print("Указанный путь не существует или не является папкой.")

    start_time = time.time()
    length = len(wheel_ets_train_x)
    for file in png_files:
        combined_mask = np.zeros((576, 352))
        image = cv2.imread(file)
        image1 = cv2.resize(image, (576, 352))
        # cv2.imshow('file', image)
        results = model(image1,
                        # imgsz=576,
                        conf=0.6,
                        show=True,
                        device='cuda',
                        verbose=False)

        combined_mask = cv2.resize(combined_mask, (128, 96))
        # time.sleep(20)
        # if results[0].masks is not None:
        #     for r0 in results[0].masks.data:
        #         combined_mask += r0.cpu().numpy()
        # combined_mask = cv2.resize(combined_mask, (128, 96))
        wheel_ets_train_x.loc[len(wheel_ets_train_x)] = combined_mask.flatten()
        
    print(len(wheel_ets_train_x) - length, 'сек:', time.time() - start_time)
cv2.waitKey()
cv2.destroyAllWindows()

Полные пути к файлам в папке:
D:\Dataset_for_autopilot\2024-04-01 08-19-11\road\2024-04-01 08-19-11_0.png 


ValueError: cannot set a row with mismatched columns

In [ ]:
len(wheel_ets_train_x)

In [ ]:
wheel_ets_train_x.to_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv', index=False)

In [ ]:
wheel_ets_train_x_ = pd.read_csv(r'C:\PycharmProjects\ETS_Autopilot\dataset_for_wheel_nn\X_train_road.csv')
len(wheel_ets_train_x_)